In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, random_split
from transformers import ViTForImageClassification
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Data - 224x224 required for ViT
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset_full = datasets.GTSRB(root='./data', split='train', transform=transform_train, download=True)
test_dataset = datasets.GTSRB(root='./data', split='test', transform=transform_test, download=True)

train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

100%|██████████| 187M/187M [00:17<00:00, 10.9MB/s] 
100%|██████████| 89.0M/89.0M [00:07<00:00, 12.6MB/s]
100%|██████████| 99.6k/99.6k [00:00<00:00, 217kB/s]


Train: 23976, Val: 2664, Test: 12630


In [4]:
# Load pretrained ViT from HuggingFace
model_vit = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=43,
    ignore_mismatched_sizes=True
)
model_vit = model_vit.to(device)
print(f"ViT loaded!")
print(f"Total parameters: {sum(p.numel() for p in model_vit.parameters()):,}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([43])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([43, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


ViT loaded!
Total parameters: 85,831,723


In [5]:
# Train ViT - 2 epochs (peak performance at epoch 2 based on prior experiments)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_vit.parameters(), lr=0.0001)

best_val_acc = 0
for epoch in range(2):
    model_vit.train()
    correct, total = 0, 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/2"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_vit(images).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = 100 * correct / total
    
    model_vit.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_vit(images).logits
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100 * val_correct / val_total
    print(f"Epoch {epoch+1}: Train={train_acc:.2f}%, Val={val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_vit.state_dict(), 'vit_best.pth')
        print(f"  → Best model saved!")

print(f"\nBest Val Accuracy: {best_val_acc:.2f}%")
print("Training complete!")

Epoch 1/2: 100%|██████████| 750/750 [14:21<00:00,  1.15s/it]


Epoch 1: Train=93.79%, Val=99.17%
  → Best model saved!


Epoch 2/2: 100%|██████████| 750/750 [14:28<00:00,  1.16s/it]


Epoch 2: Train=99.30%, Val=99.62%
  → Best model saved!

Best Val Accuracy: 99.62%
Training complete!


In [6]:
# Evaluate ViT on test set
model_vit.load_state_dict(torch.load('vit_best.pth'))
model_vit.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing ViT"):
        images, labels = images.to(device), labels.to(device)
        outputs = model_vit(images).logits
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"ViT Test Accuracy: {test_acc * 100:.2f}%")
print(f"ViT Macro F1: {test_f1:.4f}")
torch.save(model_vit.state_dict(), 'vit_final.pth')
print("ViT saved!")

Testing ViT: 100%|██████████| 395/395 [02:35<00:00,  2.54it/s]


ViT Test Accuracy: 96.66%
ViT Macro F1: 0.9173
ViT saved!
